# Retrieval test monitor

Run this notebook from top to bottom to reproduce and monitor the retrieval tests. Each test file is launched in its own subprocess with unbuffered, live output, so collection errors and long-running tests remain visible.

The notebook does not modify package code or test results. Edit `SELECTED_TESTS` to narrow the run, then rerun the execution cell as needed.

In [ ]:
from __future__ import annotations

import os
import platform
import subprocess
import sys
import time
from pathlib import Path

def find_repo_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / 'pyproject.toml').is_file() and (directory / 'tests').is_dir():
            return directory
    raise RuntimeError('Could not find the ROBERT repository root')

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
print(f'Repository: {REPO_ROOT}')
print(f'Python:     {sys.version.replace(chr(10), " " )}')
print(f'Executable: {sys.executable}')
print(f'Platform:   {platform.platform()}')

## Import preflight

This isolates package import from pytest. At the time this notebook was created, retrieval tests failed during collection at `ForwardPrediction = Spectrum | ObservedSpectrumPrediction`; this cell preserves the complete traceback if that failure is still present.

In [ ]:
preflight = subprocess.run(
    [sys.executable, '-c', 'import robert_exoplanets; print(\"Import succeeded\")'],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
)
print(preflight.stdout, end='')
if preflight.stderr:
    print(preflight.stderr, end='', file=sys.stderr)
print(f'Import exit code: {preflight.returncode}')

## Select retrieval tests

Node IDs such as `tests/test_retrieval_inference.py::test_retrieval_problem_loglike_and_oe_recover_linear_model` also work.

In [ ]:
SELECTED_TESTS = [
    'tests/test_retrieval_inference.py',
    'tests/test_retrieval_production.py',
    'tests/test_retrieval_run_config.py',
    'tests/test_runner.py',
    'tests/test_config.py',
    'tests/test_temperature_profiles.py',
    'tests/test_chemistry_profiles.py',
    'tests/test_multi_dataset.py',
]
PYTEST_ARGS = ['-vv', '-ra', '--tb=long', '-s']
print(f'{len(SELECTED_TESTS)} test targets selected')

## Run with live output

A summary is retained in `results`. Interrupt the kernel to stop the currently running test subprocess.

In [ ]:
def run_test_target(target: str) -> dict[str, object]:
    command = [sys.executable, '-m', 'pytest', *PYTEST_ARGS, target]
    print(f'\n{"=" * 88}\nRUNNING: {target}\n{"=" * 88}', flush=True)
    started = time.monotonic()
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )
    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait(timeout=10)
        print('\nTest subprocess interrupted.', flush=True)
        raise
    elapsed = time.monotonic() - started
    status = 'PASS' if returncode == 0 else 'FAIL'
    print(f'{status}: {target} ({elapsed:.2f} s)', flush=True)
    return {'target': target, 'status': status, 'returncode': returncode, 'seconds': elapsed}

results = []
suite_started = time.monotonic()
for test_target in SELECTED_TESTS:
    results.append(run_test_target(test_target))
suite_seconds = time.monotonic() - suite_started

In [ ]:
print(f'Completed {len(results)} targets in {suite_seconds:.2f} s\n')
for result in results:
    marker = '✓' if result['status'] == 'PASS' else '✗'
    print(f"{marker} {result['status']:4}  {result['seconds']:8.2f} s  {result['target']}")
failed = [result for result in results if result['returncode'] != 0]
print(f'\n{len(failed)} failed; {len(results) - len(failed)} passed')

## Optional continuous rerun

Set `RUN_CONTINUOUSLY = True` only when you want repeated monitoring. Interrupt the cell to stop.

In [ ]:
RUN_CONTINUOUSLY = False
POLL_SECONDS = 10

while RUN_CONTINUOUSLY:
    cycle_started = time.strftime('%Y-%m-%d %H:%M:%S')
    print(f'\nMonitoring cycle started {cycle_started}', flush=True)
    cycle_results = [run_test_target(target) for target in SELECTED_TESTS]
    cycle_failures = sum(result['returncode'] != 0 for result in cycle_results)
    print(f'Cycle complete: {cycle_failures} failure(s). Waiting {POLL_SECONDS} s.', flush=True)
    time.sleep(POLL_SECONDS)